# LeetCode 1221 — Split a String in Balanced Strings bằng Python

**Link đề:** https://leetcode.com/problems/split-a-string-in-balanced-strings/
**Độ khó:** Easy · **Chủ đề:** String, Greedy, Counting

```python
class Solution:
    def balancedStringSplit(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu kỹ thuật **biến đếm cân bằng (balance counter)**: `L` cộng 1, `R` trừ 1 — nền tảng cho các bài ngoặc đơn, prefix sum.
2. Nắm duyệt chuỗi, slicing `s[i:j]`, `str.count`, `itertools.accumulate`.
3. Đi từ cách thử cắt + đếm lại (`O(n²)`) → **greedy cắt ngay khi cân bằng** (`O(n)`, `O(1)` bộ nhớ).
4. Chứng minh vì sao **cắt sớm nhất có thể** cho số đoạn lớn nhất, kiểm chứng bằng brute force thử mọi cách cắt.
5. Trace, test, lỗi hay gặp, biến thể (ngoặc đơn), bài tập TODO + đáp án cuối.

## 1. Đề bài và ví dụ

Chuỗi **cân bằng (balanced)** là chuỗi có số ký tự `'L'` bằng số ký tự `'R'`.

Cho chuỗi cân bằng `s`, hãy cắt nó thành một số chuỗi con **liên tiếp** sao cho **mỗi chuỗi con đều cân bằng**. Trả về **số chuỗi con nhiều nhất** có thể.

| `s` | Kết quả | Cách cắt |
|---|:---:|---|
| `"RLRRLLRLRL"` | `4` | `"RL" | "RRLL" | "RL" | "RL"` |
| `"RLRRRLLRLL"` | `2` | `"RL" | "RRRLLRLL"` |
| `"LLLLRRRR"` | `1` | `"LLLLRRRR"` |

**Constraints:** `2 <= s.length <= 1000`; `s[i]` là `'L'` hoặc `'R'`; `s` là chuỗi cân bằng.

Chú ý ví dụ 2: không cắt được `"RL" | "RR" | "RL" | "LR" | "LL"` vì `"RR"` và `"LL"` không cân bằng — **mọi** đoạn phải cân bằng.

## 2. Nền tảng Python cần biết

### 2.1. Duyệt chuỗi, chỉ số và slicing

`s[i]` là ký tự thứ `i`; `s[i:j]` là chuỗi con từ `i` tới `j - 1`.

In [1]:
s = "RLRRLLRLRL"
print("s[0] =", s[0], "| s[2:6] =", s[2:6], "| len =", len(s))
for i, ch in enumerate(s[:4]):
    print(i, ch)

s[0] = R | s[2:6] = RRLL | len = 10
0 R
1 L
2 R
3 R


### 2.2. `str.count` — kiểm tra một đoạn có cân bằng không

In [2]:
def is_balanced(t: str) -> bool:
    return t.count("L") == t.count("R")

for t in ["RL", "RRLL", "RR", "RRRLLRLL", ""]:
    print(repr(t), "->", is_balanced(t))

'RL' -> True
'RRLL' -> True
'RR' -> False
'RRRLLRLL' -> True
'' -> True


Lưu ý: chuỗi rỗng `""` cũng có 0 `L` = 0 `R`, nhưng đề yêu cầu cắt thành các chuỗi con thật — đoạn rỗng không được tính.

### 2.3. Biến đếm cân bằng (balance)

Thay vì đếm `L` và `R` riêng, giữ **một** biến: gặp `L` → `+1`, gặp `R` → `-1`. Đoạn cân bằng ⇔ tổng thay đổi = 0.

In [3]:
balance = 0
for ch in "RRLL":
    balance += 1 if ch == "L" else -1
    print(ch, "-> balance =", balance)

R -> balance = -1
R -> balance = -2
L -> balance = -1
L -> balance = 0


`1 if ch == "L" else -1` là **biểu thức điều kiện** (ternary): trả `1` nếu đúng, `-1` nếu sai.

### 2.4. Prefix sum với `itertools.accumulate`

`accumulate([a, b, c])` → `[a, a+b, a+b+c]` — dãy **tổng tiền tố**. Balance sau mỗi ký tự chính là prefix sum của dãy `±1`.

In [4]:
from itertools import accumulate
steps = [1 if ch == "L" else -1 for ch in "RLRRLLRLRL"]
print("steps  =", steps)
print("prefix =", list(accumulate(steps)))

steps  = [-1, 1, -1, -1, 1, 1, -1, 1, -1, 1]
prefix = [-1, 0, -1, -2, -1, 0, -1, 0, -1, 0]


## 3. Cách 1 — Trực giác: cắt đoạn ngắn nhất cân bằng, kiểm tra bằng `count`

Ý nghĩ tự nhiên: đứng ở đầu phần còn lại, thử độ dài 2, 4, 6, ... — đoạn **đầu tiên** cân bằng thì cắt. Lặp lại với phần còn lại.
Kiểm tra cân bằng bằng `count` → mỗi lần kiểm tra `O(độ dài đoạn)`.

In [5]:
def split_count_naive(s: str) -> int:
    pieces = 0
    start = 0
    while start < len(s):
        end = start + 2
        while not is_balanced(s[start:end]):     # thử đoạn dài dần
            end += 2
        pieces += 1
        start = end
    return pieces

for s in ["RLRRLLRLRL", "RLRRRLLRLL", "LLLLRRRR"]:
    print(s, "->", split_count_naive(s))

RLRRLLRLRL -> 4
RLRRRLLRLL -> 2
LLLLRRRR -> 1


**Chi phí:** với chuỗi kiểu `"LLL...RRR"`, ta thử độ dài 2, 4, ..., n, mỗi lần `slice + count` tốn `O(độ dài)` → `O(n²)`. Với `n ≤ 1000` vẫn chạy, nhưng ta đang **đếm lại từ đầu** mỗi lần mở rộng đoạn.

Còn một câu hỏi lớn: **cắt sớm như vậy có thật sự cho số đoạn lớn nhất?** Ta sẽ trả lời ở mục 4 và 8.

## 4. Nút thắt và insight

**Nút thắt:** mỗi lần mở rộng đoạn, ta đếm lại `L` và `R` từ đầu đoạn. → Giữ một biến `balance` **cập nhật dần**: mở rộng 1 ký tự chỉ tốn `O(1)`.

**Insight về cắt tham lam:** gọi `bal[i]` là balance của **tiền tố** `s[0..i]`.
- Đoạn `s[a+1..b]` cân bằng ⇔ `bal[b] - bal[a] = 0` ⇔ `bal[a] == bal[b]`.
- Nếu ta cắt tại các vị trí `c1 < c2 < ... < ck = n-1` và đoạn đầu bắt đầu từ 0, thì đoạn đầu cân bằng ⇔ `bal[c1] = 0`; đoạn hai cân bằng ⇔ `bal[c2] = bal[c1] = 0`; ... Vậy **mọi điểm cắt hợp lệ là vị trí có balance tiền tố bằng 0**.
- Ngược lại, cắt tại **mọi** vị trí có balance 0 đều hợp lệ.

⇒ Số đoạn lớn nhất = **số vị trí `i` mà `bal[i] == 0`**. Greedy "cắt ngay khi balance về 0" đếm chính xác điều đó.

In [6]:
s = "RLRRLLRLRL"
bal = 0
for i, ch in enumerate(s):
    bal += 1 if ch == "L" else -1
    mark = "  <-- balance = 0, có thể cắt" if bal == 0 else ""
    print(f"i={i} ch={ch} bal={bal:+d}{mark}")

i=0 ch=R bal=-1
i=1 ch=L bal=+0  <-- balance = 0, có thể cắt
i=2 ch=R bal=-1
i=3 ch=R bal=-2
i=4 ch=L bal=-1
i=5 ch=L bal=+0  <-- balance = 0, có thể cắt
i=6 ch=R bal=-1
i=7 ch=L bal=+0  <-- balance = 0, có thể cắt
i=8 ch=R bal=-1
i=9 ch=L bal=+0  <-- balance = 0, có thể cắt


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — khởi tạo:** `balance = 0`, `count = 0`.

**Đoạn 2 — cập nhật balance:** `L` → `+1`, `R` → `-1`.

**Đoạn 3 — cắt:** nếu `balance == 0` sau khi cập nhật → vừa kết thúc một đoạn cân bằng → `count += 1`. (Không cần reset `balance` vì nó đã bằng 0.)

In [7]:
def split_greedy(s: str) -> int:
    balance = 0                      # đoạn 1
    count = 0
    for ch in s:
        if ch == "L":                # đoạn 2
            balance += 1
        else:
            balance -= 1
        if balance == 0:             # đoạn 3
            count += 1
    return count

for s in ["RLRRLLRLRL", "RLRRRLLRLL", "LLLLRRRR"]:
    print(s, "->", split_greedy(s))

RLRRLLRLRL -> 4
RLRRRLLRLL -> 2
LLLLRRRR -> 1


### Cách 3 — Prefix sum + đếm số 0

Dạng "khai báo" của cùng ý tưởng: tạo dãy balance tiền tố rồi đếm số phần tử bằng 0.

In [8]:
from itertools import accumulate

def split_prefix(s: str) -> int:
    prefix = accumulate(1 if ch == "L" else -1 for ch in s)
    return sum(1 for v in prefix if v == 0)

for s in ["RLRRLLRLRL", "RLRRRLLRLL", "LLLLRRRR"]:
    print(s, "->", split_prefix(s))

RLRRLLRLRL -> 4
RLRRRLLRLL -> 2
LLLLRRRR -> 1


### Cách 4 — Hai bộ đếm `L` và `R` riêng

Một số người thấy dễ đọc hơn: đếm `nl`, `nr`; mỗi khi `nl == nr` thì cắt.

In [9]:
def split_two_counters(s: str) -> int:
    nl = nr = 0
    count = 0
    for ch in s:
        if ch == "L":
            nl += 1
        else:
            nr += 1
        if nl == nr:
            count += 1
    return count

for s in ["RLRRLLRLRL", "RLRRRLLRLL", "LLLLRRRR"]:
    print(s, "->", split_two_counters(s))

RLRRLLRLRL -> 4
RLRRRLLRLL -> 2
LLLLRRRR -> 1


Thực ra `nl - nr` chính là `balance` — hai cách là một.

### Cách 5 — Brute force thử **mọi** cách cắt (oracle để kiểm chứng greedy)

Đệ quy: `best(start)` = số đoạn tối đa cho `s[start:]`. Thử mọi `end`, nếu `s[start:end]` cân bằng thì `1 + best(end)`. Dùng `functools.lru_cache` để nhớ kết quả (memoization), nếu không sẽ bùng nổ cấp số mũ.

In [10]:
from functools import lru_cache

def split_bruteforce(s: str) -> int:
    n = len(s)

    @lru_cache(maxsize=None)
    def best(start: int) -> int:
        if start == n:
            return 0
        result = float("-inf")            # -inf: không cắt được
        for end in range(start + 1, n + 1):
            if is_balanced(s[start:end]):
                result = max(result, 1 + best(end))
        return result

    return best(0)

for s in ["RLRRLLRLRL", "RLRRRLLRLL", "LLLLRRRR"]:
    print(s, "->", split_bruteforce(s))

RLRRLLRLRL -> 4
RLRRRLLRLL -> 2
LLLLRRRR -> 1


`@lru_cache` là **decorator**: bọc hàm `best` để mỗi `start` chỉ tính một lần rồi lưu lại.

## 6. TRACE từng bước

Ví dụ 2: `s = "RLRRRLLRLL"`

| i | ch | balance sau | = 0? | count | đoạn hiện tại |
|:---:|:---:|:---:|:---:|:---:|---|
| 0 | R | -1 | | 0 | R |
| 1 | L | 0 | ✓ cắt | 1 | **RL** |
| 2 | R | -1 | | 1 | R |
| 3 | R | -2 | | 1 | RR |
| 4 | R | -3 | | 1 | RRR |
| 5 | L | -2 | | 1 | RRRL |
| 6 | L | -1 | | 1 | RRRLL |
| 7 | R | -2 | | 1 | RRRLLR |
| 8 | L | -1 | | 1 | RRRLLRL |
| 9 | L | 0 | ✓ cắt | 2 | **RRRLLRLL** |

In [11]:
def trace_split(s: str) -> list[str]:
    balance = 0
    pieces = []
    start = 0
    for i, ch in enumerate(s):
        balance += 1 if ch == "L" else -1
        cut = balance == 0
        print(f"i={i} ch={ch} balance={balance:+d} {'-> CẮT ' + repr(s[start:i + 1]) if cut else ''}")
        if cut:
            pieces.append(s[start:i + 1])
            start = i + 1
    print("Các đoạn:", pieces, "| số đoạn:", len(pieces))
    return pieces

assert trace_split("RLRRRLLRLL") == ["RL", "RRRLLRLL"]

i=0 ch=R balance=-1 
i=1 ch=L balance=+0 -> CẮT 'RL'
i=2 ch=R balance=-1 
i=3 ch=R balance=-2 
i=4 ch=R balance=-3 
i=5 ch=L balance=-2 
i=6 ch=L balance=-1 
i=7 ch=R balance=-2 
i=8 ch=L balance=-1 
i=9 ch=L balance=+0 -> CẮT 'RRRLLRLL'
Các đoạn: ['RL', 'RRRLLRLL'] | số đoạn: 2


In [12]:
assert trace_split("RLRRLLRLRL") == ["RL", "RRLL", "RL", "RL"]

i=0 ch=R balance=-1 
i=1 ch=L balance=+0 -> CẮT 'RL'
i=2 ch=R balance=-1 
i=3 ch=R balance=-2 
i=4 ch=L balance=-1 
i=5 ch=L balance=+0 -> CẮT 'RRLL'
i=6 ch=R balance=-1 
i=7 ch=L balance=+0 -> CẮT 'RL'
i=8 ch=R balance=-1 
i=9 ch=L balance=+0 -> CẮT 'RL'
Các đoạn: ['RL', 'RRLL', 'RL', 'RL'] | số đoạn: 4


## 7. Code nộp LeetCode

### 7.1. Bản chính — greedy balance

In [13]:
class Solution:
    def balancedStringSplit(self, s: str) -> int:
        balance = 0
        count = 0
        for ch in s:
            balance += 1 if ch == "L" else -1
            if balance == 0:
                count += 1
        return count

sol = Solution()
print(sol.balancedStringSplit("RLRRLLRLRL"), sol.balancedStringSplit("RLRRRLLRLL"), sol.balancedStringSplit("LLLLRRRR"))

4 2 1


### 7.2. Bản prefix sum

In [14]:
from itertools import accumulate

class SolutionPrefix:
    def balancedStringSplit(self, s: str) -> int:
        return list(accumulate(1 if c == "L" else -1 for c in s)).count(0)

assert SolutionPrefix().balancedStringSplit("RLRRLLRLRL") == 4
print("SolutionPrefix OK")

SolutionPrefix OK


### 7.3. Bản hai bộ đếm

In [15]:
class SolutionTwoCounters:
    def balancedStringSplit(self, s: str) -> int:
        nl = nr = count = 0
        for ch in s:
            if ch == "L":
                nl += 1
            else:
                nr += 1
            if nl == nr:
                count += 1
        return count

assert SolutionTwoCounters().balancedStringSplit("LLLLRRRR") == 1
print("SolutionTwoCounters OK")

SolutionTwoCounters OK


## 8. Vì sao greedy đúng?

Gọi `bal[i]` = (số `L`) − (số `R`) trong `s[0..i]`, và `Z = {i : bal[i] = 0}`. Vì `s` cân bằng nên `n - 1 ∈ Z`.

1. **Mọi cách cắt hợp lệ chỉ cắt tại vị trí thuộc `Z`.** Giả sử các đoạn kết thúc tại `c1 < c2 < ... < ck = n-1`. Đoạn 1 cân bằng ⇒ `bal[c1] = 0`. Đoạn `t` cân bằng ⇒ `bal[ct] - bal[c(t-1)] = 0` ⇒ quy nạp `bal[ct] = 0`. Vậy `{c1..ck} ⊆ Z` ⇒ `k ≤ |Z|`.
2. **Cắt tại mọi vị trí của `Z` là hợp lệ.** Giữa hai vị trí liên tiếp của `Z`, balance tăng/giảm rồi quay về 0 ⇒ đoạn đó có số `L` = số `R`, và không rỗng.

Vậy đáp án = `|Z|` — đúng là số lần `balance == 0` mà greedy đếm. ∎

**Bất biến vòng lặp:** sau khi xử lý `s[0..i]`, `balance = bal[i]` và `count = |Z ∩ [0, i]|`.

Kiểm chứng thực nghiệm với brute force thử mọi cách cắt:

In [16]:
import random

def random_balanced(rng, half):
    chars = list("L" * half + "R" * half)
    rng.shuffle(chars)                 # trộn ngẫu nhiên tại chỗ
    return "".join(chars)

rng = random.Random(1221)
for _ in range(300):
    s = random_balanced(rng, rng.randint(1, 8))
    assert split_greedy(s) == split_bruteforce(s), s
print("Greedy == brute force trên 300 chuỗi cân bằng ngẫu nhiên")

Greedy == brute force trên 300 chuỗi cân bằng ngẫu nhiên


## 9. Độ phức tạp và so sánh

Gọi `n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Cắt ngắn nhất + `count` | `O(n²)` | `O(n)` (slice) | đếm lại từ đầu mỗi lần |
| Brute force + memo | `O(n²)` trạng thái×thử, mỗi lần `count` `O(n)` → `O(n³)` | `O(n)` | chỉ để kiểm chứng |
| **Greedy balance** | `O(n)` | `O(1)` | lời giải chuẩn |
| Prefix sum `accumulate` | `O(n)` | `O(n)` nếu tạo list | gọn |
| Hai bộ đếm | `O(n)` | `O(1)` | dễ đọc |

## 10. Bộ test

Mẫu đề, độ dài tối thiểu, xen kẽ hoàn toàn, khối lớn, và đối chiếu mọi cách với brute force.

In [17]:
impls = {
    "naive": split_count_naive,
    "greedy": split_greedy,
    "prefix": split_prefix,
    "two_counters": split_two_counters,
    "brute": split_bruteforce,
    "Solution": Solution().balancedStringSplit,
    "SolPrefix": SolutionPrefix().balancedStringSplit,
    "SolTwo": SolutionTwoCounters().balancedStringSplit,
}
cases = [
    ("RLRRLLRLRL", 4),
    ("RLRRRLLRLL", 2),
    ("LLLLRRRR", 1),
    ("LR", 1),                         # độ dài nhỏ nhất
    ("RL", 1),
    ("LRLRLRLR", 4),                   # xen kẽ: mỗi cặp là một đoạn
    ("LLRRLR", 2),
    ("RRLLRRLL", 2),
    ("LR" * 500, 500),                 # độ dài tối đa 1000
    ("L" * 500 + "R" * 500, 1),
]
for s, expected in cases:
    for name, f in impls.items():
        if name == "brute" and len(s) > 60:
            continue                   # brute force chậm với chuỗi dài
        assert f(s) == expected, (name, s[:20], f(s))
    print(f"{s[:24] + ('...' if len(s) > 24 else ''):28} -> {expected} PASS")

RLRRLLRLRL                   -> 4 PASS
RLRRRLLRLL                   -> 2 PASS
LLLLRRRR                     -> 1 PASS
LR                           -> 1 PASS
RL                           -> 1 PASS
LRLRLRLR                     -> 4 PASS
LLRRLR                       -> 2 PASS
RRLLRRLL                     -> 2 PASS
LRLRLRLRLRLRLRLRLRLRLRLR...  -> 500 PASS
LLLLLLLLLLLLLLLLLLLLLLLL...  -> 1 PASS


In [18]:
rng = random.Random(7)
for _ in range(300):
    s = random_balanced(rng, rng.randint(1, 10))
    res = {name: f(s) for name, f in impls.items()}
    assert len(set(res.values())) == 1, (s, res)
print("300 chuỗi ngẫu nhiên: mọi cách khớp nhau")

300 chuỗi ngẫu nhiên: mọi cách khớp nhau


## 11. Lỗi người mới hay gặp

### Lỗi 1: Đếm số lần ký tự **đổi** (L→R hoặc R→L)

In [19]:
def wrong_changes(s):
    return sum(1 for i in range(1, len(s)) if s[i] != s[i - 1] and s[i] == "L")

print("sai :", wrong_changes("RLRRRLLRLL"), "| đúng:", split_greedy("RLRRRLLRLL"))

sai : 3 | đúng: 2


Vì sao sai: đổi ký tự không có nghĩa là đoạn đã cân bằng (ví dụ `"RRRL"`). Phải theo dõi **chênh lệch tích luỹ**.

### Lỗi 2: Đếm số `'L'` hoặc `len(s) // 2`

Đáp án không phụ thuộc chỉ vào số ký tự mà vào **thứ tự**: `"LLRR"` và `"LRLR"` cùng 2 `L` nhưng đáp án 1 và 2.

In [20]:
for s in ["LLRR", "LRLR"]:
    print(s, "| len//2 =", len(s) // 2, "| đúng =", split_greedy(s))

LLRR | len//2 = 2 | đúng = 1
LRLR | len//2 = 2 | đúng = 2


### Lỗi 3: Kiểm tra `balance == 0` **trước** khi cập nhật

```python
for ch in s:
    if balance == 0: count += 1     # SAI: kiểm tra trước khi cộng ký tự
    balance += ...
```

Lần đầu `balance` = 0 sẵn → đếm thừa 1, và bỏ sót lần về 0 ở ký tự cuối.

In [21]:
def wrong_check_before(s):
    balance = count = 0
    for ch in s:
        if balance == 0:
            count += 1
        balance += 1 if ch == "L" else -1
    return count

for s in ["RLRRLLRLRL", "LLLLRRRR"]:
    print(s, "| sai:", wrong_check_before(s), "| đúng:", split_greedy(s))

RLRRLLRLRL | sai: 4 | đúng: 4
LLLLRRRR | sai: 1 | đúng: 1


Tình cờ cho đúng kết quả! Vì "đếm thừa ở đầu" bù đúng "bỏ sót ở cuối" (khi `s` cân bằng). Đây là **lỗi tiềm ẩn**: với biến thể "chuỗi có thể không cân bằng" nó sẽ sai:

In [22]:
print("s='RLR' (không cân bằng) | sai:", wrong_check_before("RLR"), "| greedy:", split_greedy("RLR"))

s='RLR' (không cân bằng) | sai: 2 | greedy: 1


Bài học: viết đúng thứ tự "**cập nhật trạng thái → rồi mới kiểm tra**", đừng dựa vào sự may mắn.

### Lỗi 4: Reset `balance = 0` sai chỗ hoặc thêm `abs`

Dùng `abs(balance)` hoặc chỉ đếm `L` cộng mà quên `R` trừ đều phá vỡ ý nghĩa "chênh lệch".

## 12. Mẹo nhận dạng pattern

- Hai loại ký tự "đối nhau" (`L/R`, `(`/`)`, `0/1`, `+/-`) và câu hỏi về **đoạn có số lượng bằng nhau** → biến **balance** `+1/-1`.
- "Cắt thành nhiều đoạn nhất sao cho mỗi đoạn thoả X" + X là tính chất dạng "tổng = 0" → **cắt tham lam** tại mỗi điểm tiền tố về 0.
- Hỏi "đoạn con (không nhất thiết từ đầu) có tổng 0" → **prefix sum + hash map** (gặp lại cùng giá trị prefix).

Bài cùng họ: 20 Valid Parentheses, 1021 Remove Outermost Parentheses, 1614 Maximum Nesting Depth of the Parentheses, 525 Contiguous Array, 2405/769 Max Chunks To Make Sorted.

## 13. Biến thể

### 13.1. Remove Outermost Parentheses (LeetCode 1021)

Cùng khung balance: `(` = +1, `)` = −1. Mỗi khi balance về 0 là hết một "khối nguyên thuỷ"; bỏ ngoặc đầu và cuối của mỗi khối.

In [23]:
def remove_outer(s: str) -> str:
    out = []
    depth = 0
    for ch in s:
        if ch == "(":
            if depth > 0:
                out.append(ch)
            depth += 1
        else:
            depth -= 1
            if depth > 0:
                out.append(ch)
    return "".join(out)

print(remove_outer("(()())(())"), remove_outer("()()"))

()()() 


### 13.2. Độ sâu lồng nhau lớn nhất (LeetCode 1614)

In [24]:
def max_depth(s: str) -> int:
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        elif ch == ")":
            depth -= 1
    return best

print(max_depth("(1+(2*3)+((8)/4))+1"))   # 3

3


### 13.3. Contiguous Array (LeetCode 525) — đoạn con dài nhất có số 0 bằng số 1

Không bắt buộc bắt đầu từ đầu chuỗi → nhớ **lần đầu** mỗi giá trị balance xuất hiện; gặp lại cùng giá trị ⇒ đoạn giữa có tổng 0.

In [25]:
def find_max_length(nums: list[int]) -> int:
    first = {0: -1}                 # balance 0 "xuất hiện" trước chỉ số 0
    bal = best = 0
    for i, x in enumerate(nums):
        bal += 1 if x == 1 else -1
        if bal in first:
            best = max(best, i - first[bal])
        else:
            first[bal] = i
    return best

print(find_max_length([0, 1]), find_max_length([0, 1, 0]), find_max_length([0, 0, 1, 0, 0, 0, 1, 1]))  # 2 2 6

2 2 6


## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `s = "LRRLLLRRLR"`. Viết dãy balance và cho biết số đoạn tối đa, các đoạn cụ thể.

**Bài 2 (dự đoán output).** `list(accumulate([1, -1, -1, 1]))`, `"RRLL"[1:3]`, `"LRLR".count("L")`.

**Bài 3 (điền TODO).**

In [26]:
def exercise_split(s: str) -> int:
    balance = 0
    count = 0
    for ch in s:
        # TODO 1: cập nhật balance (+1 cho 'L', -1 cho 'R')
        # TODO 2: nếu balance == 0 thì tăng count
        pass
    return count

# assert exercise_split("RLRRLLRLRL") == 4

**Bài 4 (viết lại).** Trả về **danh sách các đoạn** (không chỉ số lượng) theo cách cắt tối đa.

In [27]:
def exercise_split_pieces(s: str) -> list[str]:
    # TODO
    pass

# assert exercise_split_pieces("RLRRLLRLRL") == ["RL", "RRLL", "RL", "RL"]

**Bài 5 (biến thể).** `s` **có thể không cân bằng**. Trả số đoạn cân bằng tối đa nếu cắt **toàn bộ** `s` thành các đoạn cân bằng; nếu không thể thì trả `-1`.

In [28]:
def exercise_split_or_fail(s: str) -> int:
    # TODO
    pass

# assert exercise_split_or_fail("RLRRLLRLRL") == 4
# assert exercise_split_or_fail("RLR") == -1
# assert exercise_split_or_fail("RRL") == -1

**Bài 6 (mini challenge).** Đoạn con **liên tiếp dài nhất** của `s` (chuỗi `L/R` bất kỳ) có số `L` bằng số `R`. Trả độ dài.

In [29]:
def exercise_longest_balanced(s: str) -> int:
    # TODO
    pass

# assert exercise_longest_balanced("LLRRR") == 4
# assert exercise_longest_balanced("LLLL") == 0
# assert exercise_longest_balanced("RLRRLLRLRL") == 10

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | L = +1, R = −1 | ghi balance sau mỗi ký tự | đếm số lần bằng 0 |
| 2 | accumulate = tổng tiền tố | slicing không lấy chỉ số cuối | count đếm ký tự |
| 3 | Cập nhật rồi kiểm tra | `balance += 1 if ch == "L" else -1` | `if balance == 0: count += 1` |
| 4 | Nhớ chỉ số bắt đầu đoạn | `start` | khi balance về 0: `pieces.append(s[start:i+1]); start = i + 1` |
| 5 | Cuối cùng balance phải bằng 0 | greedy vẫn đúng nếu cắt được | `return count if balance == 0 else -1` |
| 6 | Không bắt đầu từ đầu → prefix + dict | lưu lần đầu mỗi balance | `first = {0: -1}`; `best = max(best, i - first[bal])` |

## 16. Đáp án tham khảo

**Bài 1:** balance: `+1, 0, -1, 0, +1, +2, +1, 0, +1, 0` → 4 lần về 0 → **4** đoạn: `"LR" | "RL" | "LLRR" | "LR"`.

**Bài 2:** `[1, 0, -1, 0]`, `"RL"`, `2`.

In [30]:
def answer_split(s):
    balance = count = 0
    for ch in s:
        balance += 1 if ch == "L" else -1
        if balance == 0:
            count += 1
    return count

def answer_split_pieces(s):
    pieces, start, balance = [], 0, 0
    for i, ch in enumerate(s):
        balance += 1 if ch == "L" else -1
        if balance == 0:
            pieces.append(s[start:i + 1])
            start = i + 1
    return pieces

def answer_split_or_fail(s):
    balance = count = 0
    for ch in s:
        balance += 1 if ch == "L" else -1
        if balance == 0:
            count += 1
    return count if balance == 0 else -1

def answer_longest_balanced(s):
    first = {0: -1}
    bal = best = 0
    for i, ch in enumerate(s):
        bal += 1 if ch == "L" else -1
        if bal in first:
            best = max(best, i - first[bal])
        else:
            first[bal] = i
    return best

assert answer_split_pieces("LRRLLLRRLR") == ["LR", "RL", "LLRR", "LR"]
assert (list(accumulate([1, -1, -1, 1])), "RRLL"[1:3], "LRLR".count("L")) == ([1, 0, -1, 0], "RL", 2)
assert answer_split("RLRRLLRLRL") == 4
assert answer_split_pieces("RLRRLLRLRL") == ["RL", "RRLL", "RL", "RL"]
assert answer_split_or_fail("RLRRLLRLRL") == 4
assert answer_split_or_fail("RLR") == -1 and answer_split_or_fail("RRL") == -1
assert answer_longest_balanced("LLRRR") == 4
assert answer_longest_balanced("LLLL") == 0
assert answer_longest_balanced("RLRRLLRLRL") == 10
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
balance = count = 0
for ch in s:
    balance += 1 if ch == "L" else -1   # 1) cập nhật
    if balance == 0:                     # 2) rồi mới kiểm tra
        count += 1
return count
```

**Câu thần chú:** *"Hai phe đối nhau → một biến balance. Về 0 là cắt."*

### Checklist trước khi submit
- [ ] `L` và `R` có dấu ngược nhau (+1 / −1)?
- [ ] Kiểm tra `balance == 0` **sau** khi cập nhật?
- [ ] Không đếm theo số lần đổi ký tự hay `len(s) // 2`?
- [ ] Không cần reset `balance` (đã về 0 sẵn)?
- [ ] Tên method `balancedStringSplit(self, s)`, trả `int`?